In [ ]:
from napari.utils.colormaps import Colormap
import numpy as np
import xarray as xr

import napari
import numpy as np

viewer = napari.Viewer()


napari.run()




2026-03-20 17:53:42,254 - WARNING - CuPy is not installed. GPU acceleration will not be available.
2026-03-20 17:53:42,395 - DEBUG - Updating registry with hook 'is_tomobase_phantom' (explicit=False)
2026-03-20 17:53:42,397 - DEBUG - Scanning D:\Code\Github\TimeDependentTomography\submodules\tomobase\tomobase for is_tomobase_phantom items
2026-03-20 17:53:42,399 - DEBUG - Importing tomobase.environment to check for is_tomobase_phantom items
2026-03-20 17:53:42,400 - DEBUG - Importing tomobase.log to check for is_tomobase_phantom items
2026-03-20 17:53:42,401 - DEBUG - Importing tomobase.utils to check for is_tomobase_phantom items
2026-03-20 17:53:42,434 - DEBUG - Importing tomobase.__init__ to check for is_tomobase_phantom items
2026-03-20 17:53:42,438 - DEBUG - Added category 'Deform' -> 0x3c000000 (inheritor=None)
2026-03-20 17:53:42,439 - DEBUG - Added category 'Image Processing' -> 0x40000000 (inheritor=None)
2026-03-20 17:53:42,439 - DEBUG - Added category 'Align' -> 0x44000000 (

In [ ]:

def box_edges(min_corner, max_corner):
    z0, y0, x0 = min_corner
    z1, y1, x1 = max_corner

    corners = np.array([
        [z0, y0, x0],  # 0
        [z0, y0, x1],  # 1
        [z0, y1, x1],  # 2
        [z0, y1, x0],  # 3
        [z1, y0, x0],  # 4
        [z1, y0, x1],  # 5
        [z1, y1, x1],  # 6
        [z1, y1, x0],  # 7
    ], dtype=float)

    edge_ids = [
        (0, 1), (1, 2), (2, 3), (3, 0),  # front face
        (4, 5), (5, 6), (6, 7), (7, 4),  # back face
        (0, 4), (1, 5), (2, 6), (3, 7),  # connecting edges
    ]

    return [corners[[i, j]] for i, j in edge_ids]
vertices = np.array([
    [0, 0, 0], [0, 0, 10], [0, 10, 0], [0, 10, 10],
    [10, 0, 0], [10, 0, 10], [10, 10, 0], [10, 10, 10]
])
size = 256
x,y,z = 0,0,0
lines = box_edges(min_corner=(x, y, z), max_corner=(x+size, y+size, z+size))
viewer.add_shapes(
    box,
    shape_type="path",
    edge_color="grey",
    face_color="transparent",
    edge_width=1,
)

In [7]:
colors_bop_gold = np.linspace(
    start=[0, 0, 0, 0.8],
    stop=[1, 0.84, 0, 1],
    num=10,
    endpoint=True
)
colors_gold = np.linspace(
    start=[1, 1, 1, 1],
    stop=[1, 0.84, 0, 1],
    num=10,
    endpoint=True
)
viewer.layers[0].colormap = colors_bop_gold

In [ ]:
save_folder = r'C:\Users\TCraig\Pictures\Visualizations'
viewer.screenshot(save_folder + r'\napari_visualization.png')

viewer.camera.perspective = 0


In [8]:
viewer.camera.perspective = 30

In [ ]:
save_folder = r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\Finalized_Samples\Figures\Assets\Cage-D\FFT-DIP'
sample_folder = 'cage_vol_sim'


n = viewer.layers[0].data.shape[0]

for i in range(n):
    step = list(viewer.dims.current_step)
    step[0] = i

    viewer.dims.current_step = step
    viewer.screenshot(save_folder + rf'\orthoslice_time{i}slice{step[1]}.png')



2026-03-20 16:27:25,580 - INFO - Reading VolumeTimeSeries from \\ematbyname\emat\TimC\DIPSTER-PublicationData\Cage-D\data_processing\processed_cage_dip_normalized.zarr
2026-03-20 16:27:25,581 - DEBUG - Attempting to load file \\ematbyname\emat\TimC\DIPSTER-PublicationData\Cage-D\data_processing\processed_cage_dip_normalized.zarr with {'.vmf': <function _read_vmf at 0x0000024C02B1B7E0>, '.zarr': <function _read_zarr at 0x0000024C02B342C0>, '.nc': <function _read_netcdf at 0x0000024C02B34AE0>}
2026-03-20 16:27:25,760 - DEBUG - Set context to GPUContext.NUMPY on device 0


In [ ]:
save_folder = r'C:\Users\TCraig\Pictures\Visualizations'
sample_folder = 'NS_volume'

n = viewer.layers[0].data.shape[0]

viewer.camera.perspective = 30
for i in range(n):
    step = list(viewer.dims.current_step)
    step[0] = i

    viewer.dims.current_step = step
    viewer.screenshot(save_folder + rf'\{sample_folder}\volume_time{i}.png')





In [29]:
from tomondt.data import VolumeTimeSeries

r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D.vmf'
vnd = VolumeTimeSeries.read(r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D_sage-forest-34.vmf')


def center_crop_3d(da, half_width=50):
    crop_slices = {}
    
    for dim in ["z", "y", "x"]:
        n = da.sizes[dim]
        c = n // 2
        start = max(0, c - half_width)
        stop = min(n, c + half_width)
        crop_slices[dim] = slice(start, stop)
    
    return da.isel(**crop_slices)


data = vnd.data.transpose("indices", "z", "x", "y")
data = center_crop_3d(data, half_width=50)

path = r'D:\Assets\Cage-D_processed.zarr'
name = 'Cage-D_sage_forest_34_processed'


vndt_2 = VolumeTimeSeries( name='processed_cage_dip', data=data)
'''
for i in range(data.sizes['indices']):
    if i == 0:
        subset = data.isel(indices=i).compute().values[None, ...]
        
    else:
        subset = data.isel(indices=[i]).compute()
        subset.to_zarr(vndt_2.path, append_dim='indices', mode='a')


print(vndt_2.data.shape)
'''




2026-03-17 17:40:20,116 - DEBUG - Attempting to load file \\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D_sage-forest-34.vmf with {'.vmf': <function _read_vmf at 0x000001A3B3615800>, '.zarr': <function _read_zarr at 0x000001A3B9004040>, '.nc': <function _read_netcdf at 0x000001A3B9004900>}
2026-03-17 17:40:21,840 - DEBUG - Reading VMF from \\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D_sage-forest-34.vmf with name DIP128 and times 100
2026-03-17 17:40:21,843 - DEBUG - Scheduled read of time 1.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 17:40:21,845 - DEBUG - Scheduled read of time 2.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 17:40:21,845 - DEBUG - Scheduled read of time 3.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 17:40:21,845 - DEBUG - Scheduled read of time 4.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 17:40:21,858 - DEBUG - 

"\nfor i in range(data.sizes['indices']):\n    if i == 0:\n        subset = data.isel(indices=i).compute().values[None, ...]\n\n    else:\n        subset = data.isel(indices=[i]).compute()\n        subset.to_zarr(vndt_2.path, append_dim='indices', mode='a')\n\n\nprint(vndt_2.data.shape)\n"

In [30]:
from tomondt.data import VolumeTimeSeries
import stackview

r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D.vmf'
vnd = VolumeTimeSeries.read(r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D_sage-forest-34.vmf')

def center_crop_3d(da, half_width=50):
    crop_slices = {}
    
    for dim in ["z", "y", "x"]:
        n = da.sizes[dim]
        c = n // 2
        start = max(0, c - half_width)
        stop = min(n, c + half_width)
        crop_slices[dim] = slice(start, stop)
    
    return da.isel(**crop_slices)

data = vnd.data.transpose("indices", "z", "x", "y")

data = center_crop_3d(data, half_width=50)

vnd = VolumeTimeSeries.read(r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\Volumes128\Cage-D.vmf')
vnd.data = center_crop_3d(vnd.data, half_width=50)

vndt_2 = VolumeTimeSeries( name='processed_cage_sim', data=vnd.data)

stackview.side_by_side(vnd.data.isel(indices=0).compute().values, data.isel(indices=0).compute().values)

2026-03-17 17:42:32,848 - DEBUG - Attempting to load file \\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D_sage-forest-34.vmf with {'.vmf': <function _read_vmf at 0x000001A3B3615800>, '.zarr': <function _read_zarr at 0x000001A3B9004040>, '.nc': <function _read_netcdf at 0x000001A3B9004900>}
2026-03-17 17:42:33,041 - DEBUG - Reading VMF from \\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\DIP128\Cage-D_sage-forest-34.vmf with name DIP128 and times 100
2026-03-17 17:42:33,042 - DEBUG - Scheduled read of time 1.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 17:42:33,042 - DEBUG - Scheduled read of time 2.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 17:42:33,042 - DEBUG - Scheduled read of time 3.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 17:42:33,042 - DEBUG - Scheduled read of time 4.0 with shape (128, 128, 128) and dtype <class 'numpy.float32'>
2026-03-17 17:42:33,042 - DEBUG - 

side_by_side


In [ ]:

import os

from tomondt.data import VolumeTimeSeries

path = r'D:\Code\Github\TimeDependentTomography\submodules\tomondt\tomondt\autosave'
name = 'processed_cage_dip'
name_2 = 'processed_cage_sim'

# drop the first projection from vndt.data
vndt= VolumeTimeSeries.read(os.path.join(path, name + '.zarr'))


vndt2= VolumeTimeSeries.read(os.path.join(path, name_2+ '.zarr'))
vndt2.data = vndt2.data.drop_isel(indices=0)

print(vndt.data.shape, vndt2.data.shape)

diff_plus = (vndt.data - vndt2.data).clip(min=0)
diff_minus = (vndt2.data - vndt.data).clip(min=0)

vndt_2 = VolumeTimeSeries( name='cage_plus_diff', data=diff_plus)
vndt_3 = VolumeTimeSeries( name='cage_minus_diff', data=diff_minus)



2026-03-17 18:04:58,423 - DEBUG - Attempting to load file D:\Code\Github\TimeDependentTomography\submodules\tomondt\tomondt\autosave\processed_cage_dip.zarr with {'.vmf': <function _read_vmf at 0x000001A3B3615800>, '.zarr': <function _read_zarr at 0x000001A3B9004040>, '.nc': <function _read_netcdf at 0x000001A3B9004900>}
2026-03-17 18:04:58,479 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-03-17 18:04:58,484 - DEBUG - Attempting to load file D:\Code\Github\TimeDependentTomography\submodules\tomondt\tomondt\autosave\processed_cage_sim.zarr with {'.vmf': <function _read_vmf at 0x000001A3B3615800>, '.zarr': <function _read_zarr at 0x000001A3B9004040>, '.nc': <function _read_netcdf at 0x000001A3B9004900>}
2026-03-17 18:04:58,517 - DEBUG - Set context to GPUContext.NUMPY on device 0


(100, 100, 100, 100) (100, 100, 100, 100)


2026-03-17 18:04:58,692 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-03-17 18:04:58,696 - INFO - Path D:\Code\Github\TimeDependentTomography\submodules\tomondt\tomondt\autosave\cage_plus_diff.zarr does not exist. Writing data to D:\Code\Github\TimeDependentTomography\submodules\tomondt\tomondt\autosave\cage_plus_diff.zarr.
c:\Users\TCraig\AppData\Local\miniconda3\envs\dev-tdtomo2\Lib\site-packages\zarr\api\asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
2026-03-17 18:05:03,307 - DEBUG - Attempting to load file D:\Code\Github\TimeDependentTomography\submodules\tomondt\tomondt\autosave\cage_plus_diff.zarr with {'.vmf': <function _read_vmf at 0x000001A3B3615800>, '.zarr': <function _read_zarr at 0x000001A3B9004040>, '.nc': <function _read_netcdf at 0x000001A3B9004900>}
2026-03-17 18:05:03,339 - DEBUG - Set 

In [3]:
import stackview

stackview.side_by_side(diff_plus.isel(indices=0).compute().values, diff_minus.isel(indices=0).compute().values)

NameError: name 'diff_plus' is not defined